# Milestone 08 — Responsible AI: Explainability, Subgroup Audits & Human Oversight

**Bank Account Fraud Detection — Deep Learning & MLOps**  
**Original roadmap:** Milestone 08 = Responsible AI. The previously completed inference/replay package is reused here, not retrained.

**What this notebook does:** development-only age/income/categorical cohort analysis, selection and error-rate comparisons, model explanation with SHAP, proxy-variable risk assessment, security and human-review controls, and interview preparation.

**Boundaries:** The model, Platt calibrator, preprocessing, 4.9296% economic threshold, and 5% capacity are **frozen provisional**. Month **6** is *exploratory* and has previously informed model development. Month **7 remains SEALED**. Business costs are hypothetical. No demographic race/gender data are provided; no claims about unmeasured protected-class fairness may be made.

## Success criteria and evidence

1. Use the **exact saved month-6 predictions and labels**, not new fitting or model selection.
2. Confirm row ordering against saved labels before computing any subgroup statistics.
3. Report confusion-matrix components and denominators, including **false-positive rate (FPR)** and **true-positive rate (TPR)**, for age/income and available categorical cohorts.
4. Explicitly flag low-support groups. *A zero-fraud group does not imply the system is fair*.
5. Explain the MLP with SHAP on a **small deterministic development-only sample**, with clear limitations.
6. Record data leakage/proxy/privacy/human-review risks and save **aggregate** audit reports in Git-ignored `artifacts/`.
7. No month-7 access or model/policy modifications; Milestone 07 remains open until its final one-time holdout evaluation.

## 1. Environment and frozen artifact verification

We load the existing inference class, which checks its serialized-model SHA-256 manifests and freezes the decision rule. The notebook never loads `data/raw/Base.csv`, and **must not call `.fit()` on model, preprocessor, or calibrator**.

In [ ]:
from pathlib import Path
import json
import hashlib
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "configs/milestone_07_candidate_policy_v01.json").is_file()), None)
assert ROOT is not None, "Open this notebook from the bank-account-fraud-dl-mlops repository"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.fraud_inference import FraudInference

assert Path(sys.prefix).resolve() == (ROOT / ".venv").resolve(), "Select the Bank Fraud .venv kernel"
engine = FraudInference(project_root=ROOT)  # checksum-verified; no fitting
policy = engine.policy
assert policy["status"] == "FROZEN_PROVISIONAL_CANDIDATE"
assert policy["data_protocol"]["holdout_status"] == "SEALED"
assert not policy["production_approved"]
assert abs(engine.threshold - (35 / 710)) < 1e-12
assert engine.max_fraction == 0.05
print("Model bundle: VERIFIED | policy: FROZEN PROVISIONAL")
print("Threshold:", format(engine.threshold, ".4%"), "| Capacity:", format(engine.max_fraction, ".0%"))
print("No training or fitting. Month 7 remains SEALED.")

## 2. Load development month 6 only, with exact row-alignment checks

The private `development_months_0_to_6.csv` contains months 0–6; we retain only month 6, preserving CSV row order. The notebook references previously stored sigmoid-calibrated MLP scores and compares the saved labels to the snapshot. **A mismatch stops the audit.**

In [ ]:
artifact_root = ROOT / "artifacts" / "milestone_07"
snapshot = artifact_root / "development_months_0_to_6.csv"
reference_file = artifact_root / "calibration_policy_v01/mlp_calibrated_scores.npz"
reference_labels_file = artifact_root / "restart_bundle_v01/predictions.npz"
assert snapshot.is_file() and reference_file.is_file() and reference_labels_file.is_file()

# Verify the prediction bundle checksum before reading.
cal_manifest = json.loads((artifact_root / "calibration_policy_v01/manifest.json").read_text(encoding="utf-8"))
expected_hash = cal_manifest["file_checksums"][reference_file.name]
actual_hash = hashlib.sha256(reference_file.read_bytes()).hexdigest()
assert expected_hash.lower() == actual_hash.lower(), "Prediction artifact SHA-256 mismatch"

config = engine.feature_config
model_columns = config["numeric_features"] + config["categorical_features"] + config["binary_features"]
assert len(model_columns) == 29
required_columns = list(dict.fromkeys(["month", "fraud_bool", *model_columns]))
parts = []
for chunk in pd.read_csv(snapshot, usecols=required_columns, chunksize=125_000, low_memory=False):
    assert set(chunk["month"].dropna().unique()).issubset(set(range(7))), "Out-of-bound month"
    selected = chunk.loc[chunk["month"].eq(6)]
    if len(selected):
        parts.append(selected)
month6 = pd.concat(parts, ignore_index=True)
assert len(month6) == 108_168
labels = month6["fraud_bool"].to_numpy(dtype=np.int8)
assert int(labels.sum()) == 1450

with np.load(reference_labels_file, allow_pickle=False) as saved:
    reference_labels = saved["y_m6"].copy()
with np.load(reference_file, allow_pickle=False) as saved:
    probabilities = saved["mlp_m6_sigmoid"].copy()
assert np.array_equal(labels, reference_labels), "Snapshot labels and saved predictions not aligned"
assert probabilities.shape == labels.shape
assert np.isfinite(probabilities).all() and np.all((probabilities >= 0) & (probabilities <= 1))
print(f"Month-6 rows: {len(month6):,} | Fraud: {int(labels.sum()):,}")
print("Saved prediction checksum: PASS | label alignment: PASS")
print("Month 7 accessed: NO")

## 3. Reproduce the frozen 5% investigation decision

Under the provisional policy, review an application only if its calibrated probability is above the economically derived threshold; prioritize descending fraud probability; review at most `floor(0.05 × cohort size)` applications. Ties use stable input ordering *only as a provisional development policy*.

**Never select a new threshold, capacity, or model based on subgroup outcomes in this notebook.**

In [ ]:
n = len(labels)
capacity = int(np.floor(n * engine.max_fraction))
eligible = np.flatnonzero(probabilities > engine.threshold)
ranked = eligible[np.argsort(-probabilities[eligible], kind="stable")]
chosen = ranked[:capacity]
investigate = np.zeros(n, dtype=bool)
investigate[chosen] = True
assert investigate.sum() == 5408
assert int(labels[investigate].sum()) == 705
assert not (probabilities[chosen[-1]] == probabilities[ranked[len(chosen)]]) if len(ranked) > len(chosen) else True

audit = month6[["customer_age", "income", "employment_status", "housing_status",
                "device_os", "source", "payment_type"]].copy()
audit["fraud_bool"] = labels
audit["risk_probability"] = probabilities
audit["investigate"] = investigate
print("Reviews:", int(investigate.sum()), "| Fraud caught:", int(labels[investigate].sum()))
print("Policy: UNCHANGED | Month 6: EXPLORATORY | Month 7: SEALED")

## 4. Predefined cohorts and evidence thresholds

We will **not** treat category labels such as `CA`, `BA`, or `AA` as human-interpretable demographic identities. We assess observed performance across dataset-coded cohorts.

**Minimum evidence rule** (set in advance of subgroup model-performance review): flag any group with fewer than **500 applications**, fewer than **20 actual fraud cases**, or fewer than **20 actual non-fraud cases**. These are pragmatic reporting thresholds, not statistical guarantees. Groups can be reported descriptively, but disparity conclusions for flagged groups must be qualified.

- Age: `<18`, `18–24`, `25–34`, `35–44`, `45–59`, `60+`
- Income dataset values: `<0.3`, `0.3–<0.5`, `0.5–<0.7`, `0.7–<1.0` (**not US dollars**)
- Existing categories: employment, housing, device OS, application source, payment type.

In [ ]:
audit["age_band"] = pd.cut(
    audit["customer_age"], bins=[0, 18, 25, 35, 45, 60, np.inf],
    right=False, include_lowest=True,
    labels=["<18", "18-24", "25-34", "35-44", "45-59", "60+"]
).astype("string").fillna("AGE_MISSING_OR_INVALID")
audit["income_band"] = pd.cut(
    audit["income"], bins=[-np.inf, 0.3, 0.5, 0.7, 1.0], right=False,
    labels=["<0.3", "0.3-<0.5", "0.5-<0.7", "0.7-<1.0"]
).astype("string").fillna("INCOME_MISSING_OR_INVALID")
GROUP_FIELDS = ["age_band", "income_band", "employment_status", "housing_status",
                "device_os", "source", "payment_type"]
for field in GROUP_FIELDS:
    audit[field] = audit[field].astype("string").fillna("MISSING")
print("Cohorts:", ", ".join(GROUP_FIELDS))
print("Minimum size >=500; positives >=20; negatives >=20")
print("NOTE: race/gender attributes unavailable; protected-class parity NOT established")

## 5. Subgroup confusion matrices, FPR/TPR and support

For each cohort we calculate **TP, FP, FN, TN**, base rate, selected rate, TPR (= TP / actual positives), FPR (= FP / actual negatives), and positive predictive value (= TP / all reviewed).

**Fairness metrics require denominators.** An apparent 0% TPR with zero actual fraud is undefined—not a successful result. A gap does not by itself prove discrimination or causality.

In [ ]:
def subgroup_metrics(df, field):
    rows = []
    for group, part in df.groupby(field, dropna=False, observed=True):
        y = part["fraud_bool"].to_numpy(dtype=np.int8)
        reviewed = part["investigate"].to_numpy(dtype=bool)
        tp = int(np.sum(reviewed & (y == 1)))
        fp = int(np.sum(reviewed & (y == 0)))
        fn = int(np.sum((~reviewed) & (y == 1)))
        tn = int(np.sum((~reviewed) & (y == 0)))
        n = len(part)
        pos = tp + fn
        neg = fp + tn
        sufficient = (n >= 500 and pos >= 20 and neg >= 20)
        rows.append({"Dimension": field, "Group": str(group), "N": n,
                     "Actual Fraud": pos, "Non-fraud": neg, "TP": tp, "FP": fp,
                     "FN": fn, "TN": tn, "Fraud Prevalence": pos / n,
                     "Selection Rate": (tp + fp) / n,
                     "TPR / Recall": tp / pos if pos else np.nan,
                     "FPR": fp / neg if neg else np.nan,
                     "Precision": tp / (tp + fp) if tp + fp else np.nan,
                     "Evidence": "SUFFICIENT_SCREENING" if sufficient else "LOW_SUPPORT"})
    return pd.DataFrame(rows)

subgroup_results = pd.concat([subgroup_metrics(audit, f) for f in GROUP_FIELDS], ignore_index=True)
assert subgroup_results["N"].gt(0).all()
assert (subgroup_results["TP"] + subgroup_results["FP"] +
        subgroup_results["FN"] + subgroup_results["TN"] == subgroup_results["N"]).all()
for field in GROUP_FIELDS:
    subset = subgroup_results[subgroup_results["Dimension"].eq(field)]
    assert int(subset["N"].sum()) == n
    assert int(subset["Actual Fraud"].sum()) == 1450
    assert int(subset["TP"].sum()) == 705

display(subgroup_results.round(5))
print("Groups marked LOW_SUPPORT:", int(subgroup_results["Evidence"].eq("LOW_SUPPORT").sum()))
print("Step 08-RA subgroup arithmetic: PASS")

## 6. Screening-level disparity summaries

These are **descriptive gaps**, not hypothesis-tested or causal fairness findings. Only groups passing the minimum-evidence rule contribute to the max-minus-min screening gaps. No subgroup-based thresholds will be introduced.

In [ ]:
gap_rows = []
for dimension, part in subgroup_results.groupby("Dimension"):
    supported = part.loc[part["Evidence"].eq("SUFFICIENT_SCREENING")]
    row = {"Dimension": dimension, "Supported Groups": len(supported),
           "Excluded Low-Support Groups": len(part) - len(supported)}
    for metric in ["Selection Rate", "FPR", "TPR / Recall", "Precision"]:
        vals = supported[metric].dropna()
        row[f"{metric} gap (pp)"] = 100 * (vals.max() - vals.min()) if len(vals) >= 2 else np.nan
    gap_rows.append(row)
disparity_screen = pd.DataFrame(gap_rows)
display(disparity_screen.round(3))
print("Warning: descriptive gaps are not legal fairness findings or statistical tests.")

### Screening-level disparity Summary
The three most important findings are:
- Housing status: The largest FPR gap, 17.186 percentage points, and the largest recall gap, 59.775 percentage points.
- Device operating system: An FPR gap of 11.247 percentage points, suggesting substantially different investigation burdens among legitimate applicants.
- Age: An FPR gap of 9.404 percentage points, which warrants closer examination.

**Application source:** Only one group met the evidence requirements. Therefore, its disparity metrics correctly show NaN—there are not two sufficiently supported groups to compare.

## 7. Visualize subgroup error rates

A display of one cohort's FPR and TPR can help identify questions for further review. Avoid visually interpreting low-support groups as reliable conclusions.

In [ ]:
dimension_to_plot = "age_band"  # change the display only; do NOT tune model or policy
plotted = subgroup_results[subgroup_results["Dimension"].eq(dimension_to_plot)].copy()
plot_data = plotted.set_index("Group")[["FPR", "TPR / Recall"]] * 100
ax = plot_data.plot(kind="bar", figsize=(10, 4), rot=25)
ax.set_title(f"Exploratory FPR and TPR by {dimension_to_plot} (month 6)")
ax.set_ylabel("Percent")
ax.set_xlabel("Dataset-coded group")
plt.tight_layout()
plt.show()
display(plotted[["Group", "N", "Actual Fraud", "Evidence"]])

## Sub-group Error Rate Summary
Positive finding: The model identifies approximately 70% of confirmed fraud cases among applicants aged 60+, compared with 32% among applicants aged 18–24.

Concern: The model also selects a much larger proportion of legitimate applicants aged 60+ for investigation.

This illustrates an important trade-off: higher fraud detection can coincide with a greater false-positive burden.

However, the age groups have different underlying fraud prevalence rates, so the chart alone cannot establish whether the differences are unfair or why they occur.

**Important limitation:** The under-18 group contains only nine confirmed fraud cases. Its 0% recall is not sufficient evidence of a reliable performance pattern.

## 8. Explainability with SHAP — scope and limitations

The input to the MLP is the **55-feature transformed representation** (including one-hot indicators and sentinel flags). We use SHAP only on the **existing frozen MLP**. For speed, we explain a deterministic small sample of **64 month-6 applications** using **32 month-6 background applications**.

**Key interpretation:** The explanations below are for **raw model logits**, *not calibrated probabilities*. The saved sigmoid calibrator is a separate monotonic transformation. A SHAP attribution is **not proof of a causal factor**, and correlations and one-hot encoding can complicate interpretation.

SHAP is required for this section. If `import shap` fails, install a compatible version in the project's `.venv`, then rerun this cell; **do not reinstall PyTorch or rerun training**.

In [ ]:
import importlib.util
print("SHAP available:", importlib.util.find_spec("shap") is not None)
print("PyTorch available:", importlib.util.find_spec("torch") is not None)
print("If SHAP is unavailable, stop here and resolve only that dependency.")

In [ ]:
import shap
import torch
from src.features.preprocessing import prepare_model_features

rng = np.random.default_rng(42)
idx = rng.choice(len(month6), size=96, replace=False)
background_idx = idx[:32]
explain_idx = idx[32:]
prepared = prepare_model_features(month6.loc[:, model_columns], config=config)
X_55 = np.asarray(engine.preprocessor.transform(prepared), dtype=np.float32)
assert X_55.shape == (108_168, 55)
assert np.isfinite(X_55).all()
transformed_names = engine.preprocessor.get_feature_names_out()
assert len(transformed_names) == 55
engine.model.eval()
background_t = torch.from_numpy(X_55[background_idx])
explain_t = torch.from_numpy(X_55[explain_idx])

# Adapt the model output shape for SHAP without changing its weights.
class SHAPLogitAdapter(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, features):
        logits = self.model(features)
        return logits.reshape(-1, 1)


shap_model = SHAPLogitAdapter(engine.model).eval()

explainer = shap.DeepExplainer(shap_model, background_t)

shap_result = explainer.shap_values(explain_t)
if isinstance(shap_result, list):
    shap_result = shap_result[0]
shap_values = np.asarray(shap_result)
if shap_values.ndim == 3 and shap_values.shape[-1] == 1:
    shap_values = shap_values[..., 0]
assert shap_values.shape == (64, 55), f"Unexpected SHAP shape: {shap_values.shape}"
assert np.isfinite(shap_values).all()
print("SHAP explanation target: MLP raw logits")
print("Background applications:", len(background_idx), "| Explained applications:", len(explain_idx))
print("SHAP shape:", shap_values.shape, "| Holdout touched: NO")

## 9. Review global SHAP contribution magnitudes

Mean absolute SHAP measures average *magnitude* of impact on raw-logit output over this small exploratory sample; it does not measure effect direction, financial impact, or causal importance.

In [ ]:
global_shap = pd.DataFrame({
    "Transformed feature": transformed_names,
    "Mean absolute SHAP (logit units)": np.abs(shap_values).mean(axis=0),
}).sort_values("Mean absolute SHAP (logit units)", ascending=False)
display(global_shap.head(20))
ax = global_shap.head(15).sort_values("Mean absolute SHAP (logit units)").plot.barh(
    x="Transformed feature", y="Mean absolute SHAP (logit units)",
    legend=False, figsize=(9, 6))
ax.set_title("MLP logit explanation: top transformed features (64 development cases)")
ax.set_xlabel("Mean absolute SHAP")
plt.tight_layout()
plt.show()

## 10. Proxy-variable and fairness limitations register

Potential risk-sensitive feature families include `customer_age`, `income`, `housing_status`, `employment_status`, `device_os`, `source`, and geographic proxies such as `zip_count_4w` (not individual ZIP codes). These variables may correlate with attributes not present in this dataset. **Do not claim demographic fairness or disparate-impact compliance without appropriate protected-attribute data, measurement design, review, and legal guidance.**

For stakeholder review: distinguish protected-class fairness, outcomes-based subgroup disparities, sampling bias, model explanation, and causal discrimination. A low-support group may deserve *more investigation*, not exclusion from concern.

## 11. Security and privacy review

| Control | Current evidence | Status / remaining work |
|---|---|---|
| Target leakage protection | `fraud_bool`, `month`, `device_fraud_count` excluded from model features | Implemented in existing preprocessing |
| Artifact integrity | SHA-256 checks in `FraudInference` | Implemented locally |
| Trusted artifact source | `joblib` deserialization may execute code | **Trust requirement**; add authenticated, secure retrieval in later milestone |
| PII handling | No personal names/addresses logged by notebook | **Policy required** for eventual API logs, retention, redaction |
| Data access | Development-only month-6 snapshot | Verified; month 7 sealed |
| API authentication, rate limits and transport security | No API exists yet | Pending original Milestone 09 |
| Drift / monitoring | No deployed monitoring yet | Pending Milestone 11 |
| Decision governance | Model only recommends review | Human-review procedure required; automatic punitive decisions prohibited |

**Security conclusion:** Research safeguards are in place; production security has **not** been verified.

## 12. Human-review and escalation policy — draft

**Policy status: DRAFT / NOT PRODUCTION APPROVED**

1. The model **prioritizes review**; it does **not** automatically deny an account or impose adverse customer action.
2. Within a defined complete decision cohort, apply the frozen `p > 4.9296%` economic eligibility condition, rank descending by calibrated risk, and cap at 5%. Hypothetical costs must be approved before real use.
3. The assigned analyst reviews legitimate case evidence, documents a reasoned decision, and is empowered to override the model.
4. Flag disputed cases and suspected false positives for a separate second review, with escalation and appeal pathways.
5. Audit outcomes by cohorts where sufficiently supported; document uncertainty and unavailable protected-attribute coverage.
6. Prohibit sensitive data in logs; restrict access to scores, artifacts, and decisions; define retention/incident procedures.
7. Escalate persistent disparity, unusual drift, or data-quality incidents; pause automation where risks are material.
8. Do not modify the model or threshold based on month-7 holdout results. Any subsequent model version requires a fresh governance and evaluation plan.

## 13. Save **aggregate-only** local audit outputs and manifest

The files remain under Git-ignored `artifacts/milestone_08_responsible_ai/`; they are **not** uploaded to GitHub. Save only cohort-level metrics, screening gaps, and explanatory feature names/magnitudes—not individual decisions, labels, or predictions.

In [ ]:
OUT_DIR = ROOT / "artifacts/milestone_08_responsible_ai"
OUT_DIR.mkdir(parents=True, exist_ok=True)
subgroup_results.to_csv(OUT_DIR / "m08ra_subgroup_metrics_v01.csv", index=False)
disparity_screen.to_csv(OUT_DIR / "m08ra_disparity_screen_v01.csv", index=False)
global_shap.to_csv(OUT_DIR / "m08ra_global_shap_v01.csv", index=False)
summary_manifest = {
    "milestone": "08-responsible-ai", "data_month": 6, "status": "EXPLORATORY",
    "holdout_month": 7, "holdout_status": "SEALED", "n": int(len(labels)),
    "fraud_cases": int(labels.sum()), "selected": int(investigate.sum()),
    "fraud_caught": int(labels[investigate].sum()),
    "group_min_n": 500, "group_min_actual_positives": 20,
    "group_min_actual_negatives": 20,
    "shap_background": 32, "shap_explained": 64,
    "model_modified": False, "policy_modified": False,
    "race_or_gender_fairness_measured": False,
}
(OUT_DIR / "m08ra_audit_manifest_v01.json").write_text(json.dumps(summary_manifest, indent=2), encoding="utf-8")
print("Aggregate outputs saved privately:", OUT_DIR)
print("Holdout: SEALED | Model & policy: unchanged")

## Milestone 08 — Responsible AI: Notebook Summary

## Project Overview

**Project:** Bank Account Fraud Detection — Deep Learning & MLOps

**Objective:** Evaluate the explainability, subgroup performance, fairness limitations, security risks, and human oversight requirements of the previously trained fraud-detection system.

The notebook reuses the frozen weighted PyTorch MLP, fitted preprocessing pipeline, and sigmoid (Platt) calibrator. It does not retrain or modify the model.

## Dataset and Evaluation Protocol

| Component | Configuration |
|---|---|
| Model training | Months 0–4 |
| Probability calibration | Month 5 |
| Responsible AI evaluation | Month 6 — exploratory |
| Final holdout | Month 7 — sealed |
| Evaluation applications | 108,168 |
| Confirmed fraud cases | 1,450 |
| Frozen investigation threshold | 4.9296% |
| Maximum investigation capacity | 5% |

The notebook verifies the saved prediction artifact and checks label alignment before calculating subgroup statistics.

## 1. Subgroup Performance and Fairness Screening

The notebook analyzes the following available attributes: customer age, dataset-coded income, employment status, housing status, device operating system, application source, and payment type.

For each subgroup, it calculates true positives, false positives, false negatives, true negatives, fraud prevalence, selection rate, true-positive rate (recall), false-positive rate, and precision.

Groups with fewer than 500 applications, 20 confirmed fraud cases, or 20 non-fraud cases are flagged as having insufficient support for the defined screening analysis.

Disparity gaps are descriptive and do not establish discrimination, causality, or legal compliance.

## 2. Model Explainability — SHAP

The notebook uses SHAP DeepExplainer with the existing PyTorch MLP.

The explanation configuration uses 32 background applications and 64 explained applications from month 6, with a deterministic random seed.

Explanations apply to the model's 55 transformed features and its raw logits, not its calibrated fraud probabilities.

Mean absolute SHAP values are used to summarize feature contribution magnitudes. These explanations are not causal evidence.

## 3. Proxy Variables and Fairness Limitations

The analysis documents potential proxy-variable risks involving age, income, housing, employment, device characteristics, application source, and geographic-related features.

Race and gender are not included in the available dataset. Consequently, the notebook cannot establish fairness across those demographic groups.

## 4. Security and Privacy Assessment

The security review covers leakage safeguards, model artifact integrity, trusted serialized artifacts, personally identifiable information, access controls, logging, and future deployment safeguards.

Existing safeguards are distinguished from controls that remain unimplemented or require further review.

## 5. Human Oversight and Escalation

A draft human-review policy specifies that model scores prioritize investigations rather than automatically rejecting applications.

It addresses analyst review, documented decisions, human overrides, second-level review, appeals, privacy safeguards, and escalation of suspected fairness or operational issues.

This policy is a draft and is not approved for production.

## 6. Reproducible Audit Deliverables

The notebook is designed to save aggregate subgroup metrics, disparity screening results, SHAP contribution summaries, and an audit manifest under:

`artifacts/milestone_08_responsible_ai/`

Individual application predictions and records are not included in those saved aggregate reports.

## 7. Learning and Interview Preparation

The notebook includes two practice exercises with worked solutions and ten interview questions covering fairness metrics, subgroup support, SHAP interpretation, leakage prevention, evaluation integrity, and human oversight.

## Conclusion and Outstanding Work

Milestone 08 provides a framework for examining how the frozen fraud model behaves across available subgroups, understanding its predictions, and documenting responsible-use limitations.

Completion still requires successful notebook execution, examination of subgroup and SHAP outputs, interpretation of findings, and verification of saved audit artifacts.

**Milestone 07 remains open until the frozen model and policy undergo their one-time independent evaluation on the sealed month-7 holdout.**

## Ten interview / knowledge-check questions

1. Why does month 6 remain exploratory?
2. What is the difference between **false-positive rate** and **precision**?
3. Why do we need subgroup confusion-matrix denominators?
4. Why does a zero-positive cohort have undefined TPR?
5. Can strong overall PR-AUC guarantee fairness?
6. How does the 5% capacity restriction influence subgroup review selection?
7. What does SHAP explain in this notebook: logits or calibrated probabilities?
8. Why is `device_fraud_count` excluded?
9. Why must `joblib` files come from trusted sources?
10. What happens if the final month-7 evaluation is disappointing?

### Interview answer guide

1. Month 6 influenced model development; repeated evaluation is not an untouched estimate.
2. FPR = FP/(FP+TN), while precision = TP/(TP+FP).
3. Group sizes, fraud counts and negative counts affect stability and interpretation.
4. It divides by zero actual fraud cases and is mathematically undefined.
5. No; ranking quality is not a group-parity guarantee.
6. Selection is based on global risk ranking, so groups can have different selection/TPR/FPR even with one universal threshold.
7. Existing weighted MLP **raw logits**, before Platt probability calibration.
8. It was excluded by the project's documented leakage-risk policy.
9. Loading a malicious pickle/joblib file can execute arbitrary Python code.
10. Report it once, do not retune on month 7, and plan any new model against a newly designated untouched test set.